In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import FloatSlider, IntSlider, RadioButtons, VBox, HBox, HTML, interactive_output, Layout, GridBox
from IPython.display import display

# ============================================================
# RENEWAL AND WHITENING FILTERS
# ============================================================

# ============================================================
# CSS FOR HORIZONTAL RADIO BUTTONS
# ============================================================

display(HTML("""
<style>
.rw-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    align-items: center !important;
    gap: 18px !important;
}
.rw-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
}
.rw-radio > label {
    display: none !important;
}
</style>
"""))

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.40;
    width:1080px;
    margin-bottom:8px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#8a4b08;
    margin-bottom:8px;
">
Renewal and Whitening Filters
</div>

<div style="margin-bottom:4px;">
<b>Renewal filter:</b> transforms white noise w[n] into a correlated WSS process x[n].
</div>

<div style="margin-bottom:4px;">
<b>Whitening filter:</b> W(z) = 1/I(z) removes the correlation and recovers a white-noise sequence.
</div>

<div style="margin-bottom:4px;">
AR, MA and ARMA processes correspond to different forms of the renewal filter I(z).
</div>

<div>
<b>This notebook:</b> verifies the inverse relation I(z)W(z) = 1 in both the time and frequency domains.
</div>

</div>
""")

# ============================================================
# RADIO BUTTONS
# ============================================================

type_selector = RadioButtons(
    options=['AR', 'MA', 'ARMA'],
    value='AR',
    description='',
    layout=Layout(width='235px')
)

type_selector.add_class('rw-radio')

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}
slider_layout = Layout(width='145px')

a_slider = FloatSlider(
    min=0.0,
    max=0.90,
    step=0.05,
    value=0.75,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

b_slider = FloatSlider(
    min=-0.85,
    max=0.85,
    step=0.05,
    value=0.45,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

sigma_slider = FloatSlider(
    min=0.5,
    max=2.0,
    step=0.1,
    value=1.0,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

N_slider = IntSlider(
    min=500,
    max=5000,
    step=500,
    value=2500,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# CURRENT VALUES
# ============================================================

a_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.75</div>'
)

b_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.45</div>'
)

sigma_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.0</div>'
)

N_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">2500</div>'
)

# ============================================================
# UPDATE VALUES
# ============================================================

def update_a_value(change):
    a_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{a_slider.value:.2f}</div>'

def update_b_value(change):
    b_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{b_slider.value:.2f}</div>'

def update_sigma_value(change):
    sigma_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{sigma_slider.value:.1f}</div>'

def update_N_value(change):
    N_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{N_slider.value}</div>'

a_slider.observe(update_a_value, names='value')
b_slider.observe(update_b_value, names='value')
sigma_slider.observe(update_sigma_value, names='value')
N_slider.observe(update_N_value, names='value')

# ============================================================
# ENABLE / DISABLE PARAMETERS
# ============================================================

def update_controls(change=None):

    if type_selector.value == 'AR':

        a_slider.disabled = False
        b_slider.disabled = True

    elif type_selector.value == 'MA':

        a_slider.disabled = True
        b_slider.disabled = False

    else:

        a_slider.disabled = False
        b_slider.disabled = False

type_selector.observe(update_controls, names='value')

update_controls()

# ============================================================
# CONTROL LABELS
# ============================================================

type_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Process type:</div>'
)

a_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">AR coefficient a:</div>'
)

b_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">MA coefficient b:</div>'
)

sigma_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Noise std σw:</div>'
)

N_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Samples N:</div>'
)

empty_value = HTML('<div></div>')

# ============================================================
# CONTROLS GRID
# ============================================================

controls_grid = GridBox(
    children=[
        type_label, type_selector, empty_value,
        a_label, a_slider, a_value,
        b_label, b_slider, b_value,
        sigma_label, sigma_slider, sigma_value,
        N_label, N_slider, N_value
    ],
    layout=Layout(
        width='400px',
        grid_template_columns='130px 180px 55px',
        grid_template_rows='34px 34px 34px 34px 34px',
        grid_gap='4px 6px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#8a4b08;
            margin-bottom:6px;
        ">
        Filter Parameters
        </div>
        """),
        controls_grid
    ],
    layout=Layout(
        width='425px',
        min_width='425px',
        padding='12px 14px',
        border='1px solid #d8c1a8',
        overflow='hidden',
        margin='16px 0px 0px 12px'
    )
)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# MAIN FUNCTION
# ============================================================

def plot_renewal_whitening(process_type='AR', a=0.75, b=0.45, sigma_w=1.0, N=2500):

    # --------------------------------------------------------
    # WHITE-NOISE INPUT
    # --------------------------------------------------------

    rng = np.random.default_rng(51)

    w = rng.normal(0.0, sigma_w, N)

    # --------------------------------------------------------
    # RENEWAL FILTER I(z)
    # --------------------------------------------------------

    if process_type == 'AR':

        b_I = np.array([1.0])
        a_I = np.array([1.0, -a])

        renewal_text = f'I(z) = 1 / (1 - {a:.2f} z⁻¹)'

    elif process_type == 'MA':

        b_I = np.array([1.0, b])
        a_I = np.array([1.0])

        renewal_text = f'I(z) = 1 + {b:.2f} z⁻¹'

    else:

        b_I = np.array([1.0, b])
        a_I = np.array([1.0, -a])

        renewal_text = f'I(z) = (1 + {b:.2f} z⁻¹) / (1 - {a:.2f} z⁻¹)'

    # --------------------------------------------------------
    # COLORED PROCESS
    # --------------------------------------------------------

    x = signal.lfilter(b_I, a_I, w)

    # --------------------------------------------------------
    # WHITENING FILTER
    #
    # W(z) = 1 / I(z)
    # --------------------------------------------------------

    b_W = a_I.copy()
    a_W = b_I.copy()

    w_hat = signal.lfilter(b_W, a_W, x)

    # --------------------------------------------------------
    # REMOVE INITIAL TRANSIENT
    # --------------------------------------------------------

    transient = min(300, N // 10)

    w_ss = w[transient:]
    x_ss = x[transient:]
    w_hat_ss = w_hat[transient:]

    # ========================================================
    # NORMALIZED AUTOCORRELATION
    # ========================================================

    max_lag = 30

    lags = np.arange(-max_lag, max_lag + 1)

    def normalized_acf(sequence):

        sequence = sequence - np.mean(sequence)

        corr = np.correlate(sequence, sequence, mode='full')

        center = len(corr) // 2

        values = corr[center - max_lag:center + max_lag + 1]

        normalization = len(sequence) - np.abs(lags)

        values = values / normalization

        if values[max_lag] != 0:
            values = values / values[max_lag]

        return values

    R_x = normalized_acf(x_ss)

    R_white = normalized_acf(w_hat_ss)

    # ========================================================
    # PSD ESTIMATION
    # ========================================================

    nperseg = min(512, len(x_ss))

    f_x, P_x = signal.welch(
        x_ss,
        fs=2.0 * np.pi,
        window='hann',
        nperseg=nperseg,
        noverlap=nperseg // 2,
        return_onesided=False,
        scaling='density'
    )

    f_w, P_white = signal.welch(
        w_hat_ss,
        fs=2.0 * np.pi,
        window='hann',
        nperseg=nperseg,
        noverlap=nperseg // 2,
        return_onesided=False,
        scaling='density'
    )

    f_x = np.fft.fftshift(f_x)

    f_w = np.fft.fftshift(f_w)

    P_x = np.fft.fftshift(P_x)

    P_white = np.fft.fftshift(P_white)

    P_x = 2.0 * np.pi * P_x

    P_white = 2.0 * np.pi * P_white

    # ========================================================
    # THEORETICAL PSD
    # ========================================================

    omega = np.linspace(-np.pi, np.pi, 1600)

    _, H_I = signal.freqz(b_I, a_I, worN=omega)

    S_x_theory = sigma_w**2 * np.abs(H_I)**2

    white_level = sigma_w**2 * np.ones_like(omega)

    # ========================================================
    # SINGLE FIGURE
    # ========================================================

    fig = plt.figure(figsize=(8.2, 6.8))

    gs = fig.add_gridspec(2, 2, hspace=0.48, wspace=0.31)

    ax1 = fig.add_subplot(gs[0, 0])

    ax2 = fig.add_subplot(gs[0, 1])

    ax3 = fig.add_subplot(gs[1, 0])

    ax4 = fig.add_subplot(gs[1, 1])

    # ========================================================
    # GRAPH 1:
    # WHITE-NOISE INPUT
    # ========================================================

    show_N = min(350, len(w_ss))

    ax1.plot(np.arange(show_N), w_ss[:show_N], linewidth=0.9)

    ax1.set_xlim(0, show_N - 1)

    ax1.set_ylim(-7.0, 7.0)

    ax1.set_xlabel('Time index n', fontsize=10)

    ax1.set_ylabel('w[n]', fontsize=10)

    ax1.set_title('White-Noise Input', fontsize=12, pad=8)

    ax1.tick_params(axis='both', labelsize=9)

    ax1.grid(True, linestyle=':', alpha=0.5)

    # ========================================================
    # GRAPH 2:
    # COLORED PROCESS
    # ========================================================

    ax2.plot(np.arange(show_N), x_ss[:show_N], linewidth=1.0)

    ax2.set_xlim(0, show_N - 1)

    ax2.set_ylim(-15.0, 15.0)

    ax2.set_xlabel('Time index n', fontsize=10)

    ax2.set_ylabel('x[n]', fontsize=10)

    ax2.set_title(f'Output of the {process_type} Renewal Filter', fontsize=12, pad=8)

    ax2.tick_params(axis='both', labelsize=9)

    ax2.grid(True, linestyle=':', alpha=0.5)

    # ========================================================
    # GRAPH 3:
    # AUTOCORRELATION
    # ========================================================

    ax3.plot(lags, R_x, linewidth=1.8, label='Colored process')

    ax3.plot(lags, R_white, linewidth=1.8, label='After whitening')

    ax3.axhline(0, linewidth=0.8)

    ax3.axvline(0, linewidth=0.8, linestyle=':')

    ax3.set_xlim(-max_lag, max_lag)

    ax3.set_ylim(-0.35, 1.15)

    ax3.set_xlabel('Lag k', fontsize=10)

    ax3.set_ylabel('Normalized autocorrelation', fontsize=10)

    ax3.set_title('Autocorrelation Before and After Whitening', fontsize=12, pad=8)

    ax3.tick_params(axis='both', labelsize=9)

    ax3.grid(True, linestyle=':', alpha=0.5)

    ax3.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.20),
        ncol=2,
        fontsize=8
    )

    # ========================================================
    # GRAPH 4:
    # PSD
    # ========================================================

    ax4.plot(
        omega,
        S_x_theory,
        linewidth=2.0,
        label='Theoretical colored PSD'
    )

    ax4.plot(
        f_x,
        P_x,
        linewidth=1.0,
        alpha=0.65,
        label='Estimated colored PSD'
    )

    ax4.plot(
        omega,
        white_level,
        linestyle='--',
        linewidth=2.0,
        label='Ideal white PSD'
    )

    ax4.plot(
        f_w,
        P_white,
        linewidth=1.1,
        alpha=0.80,
        label='Estimated whitened PSD'
    )

    ax4.set_xlim(-np.pi, np.pi)

    ax4.set_xticks([
        -np.pi,
        -np.pi / 2,
        0,
        np.pi / 2,
        np.pi
    ])

    ax4.set_xticklabels([
        '-π',
        '-π/2',
        '0',
        'π/2',
        'π'
    ])

    ax4.set_xlabel('Angular frequency ω', fontsize=10)

    ax4.set_ylabel('PSD', fontsize=10)

    ax4.set_title('Power Spectral Density', fontsize=12, pad=8)

    ax4.tick_params(axis='both', labelsize=9)

    ax4.grid(True, linestyle=':', alpha=0.5)

    ax4.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.20),
        ncol=2,
        fontsize=7.5
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.09,
        right=0.97,
        top=0.93,
        bottom=0.16
    )

    plt.show()

    plt.close(fig)

    # ========================================================
    # NUMERICAL VERIFICATION
    # ========================================================

    reconstruction_error = np.sqrt(
        np.mean(
            (w_hat_ss - w_ss)**2
        )
    )

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.42;
        width:760px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Renewal filter:</b>
    {renewal_text}

    <br>

    <b>Whitening filter:</b>
    W(z) = 1 / I(z)

    &nbsp;&nbsp;&nbsp;

    <b>RMS reconstruction error:</b>
    {reconstruction_error:.6f}

    </div>
    """

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

output = interactive_output(
    plot_renewal_whitening,
    {
        'process_type': type_selector,
        'a': a_slider,
        'b': b_slider,
        'sigma_w': sigma_slider,
        'N': N_slider
    }
)

# ============================================================
# GRAPH AREA
# ============================================================

graph_area = VBox(
    [
        output,
        result_html
    ],
    layout=Layout(
        width='800px',
        min_width='800px',
        overflow='hidden'
    )
)

# ============================================================
# MAIN BODY
#
# GRAPHS LEFT
# FILTER PARAMETERS RIGHT
# ============================================================

body_layout = HBox(
    [
        graph_area,
        controls_card
    ],
    layout=Layout(
        width='1240px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='hidden'
    )
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.42;
    width:1080px;
    padding:11px 15px;
    border:1px solid #dfcdb9;
    background:#fffaf4;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#8a4b08;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The renewal filter transforms flat-spectrum white noise into a correlated process with a non-flat PSD.
</div>

<div style="margin-bottom:4px;">
The inverse whitening filter removes the correlation: its autocorrelation becomes concentrated near k = 0 and its PSD becomes approximately flat.
</div>

<div>
Because W(z) = 1/I(z), cascading the renewal and whitening filters ideally reproduces the original white-noise sequence.
</div>

</div>
""")

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        body_layout,
        interpretation
    ],
    layout=Layout(
        width='1240px',
        overflow='hidden'
    )
)

display(main_layout)